# Imports, paths, config

In [1]:
# =========================================================
# Fast Reading (Digest & Focus) — DATA PREP (Kaggle)
# - CNN/DailyMail (CSV)
# - XL-Sum (JSONL by language)
# - MLSUM-FR (CSV)
# Outputs:
#   /kaggle/working/corpus_for_demo.json
#   /kaggle/working/eval_pairs.json
# =========================================================

import os, re, json, random
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

# -----------------------------
# 0) Paths (as in your screenshots)
# -----------------------------
P_CNNDM   = Path("/kaggle/input/newspaper-text-summarization-cnn-dailymail/cnn_dailymail")
P_XLSUM   = Path("/kaggle/input/xl-sum/XLSum_complete_v2.0")
P_MLSUM   = Path("/kaggle/input/mlsam-multilingual-summarization-dataset")

# pick the languages you want tonight; keep it simple: EN + FR
XLSUM_LANGS = {
    "en": "english",  # file stems like english_train.jsonl
    "fr": "french"
}

# target sample sizes (adjust if RAM tight)
N_CNNDM   = 2000
N_XLSUM   = 2000   # total across the chosen language(s); split below
N_MLSUMFR = 2000

# keep only these languages downstream
KEEP_LANGS = {"en", "fr"}

In [25]:
# Cell 1 — Setup & Config

import os, re, json, random
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

# Paths (exactly like your Kaggle inputs)
P_CNNDM = Path("/kaggle/input/newspaper-text-summarization-cnn-dailymail/cnn_dailymail")
P_XLSUM = Path("/kaggle/input/xl-sum/XLSum_complete_v2.0")
P_MLSUM = Path("/kaggle/input/mlsam-multilingual-summarization-dataset")

# We’ll keep EN + FR tonight
XLSUM_LANGS = {"en": "english", "fr": "french"}

# Target sample sizes (tune if RAM is tight)
N_CNNDM   = 3000
N_XLSUM   = 3000   # across chosen languages
N_MLSUMFR = 3000

KEEP_LANGS = {"en", "fr"}


# Helper functions

In [26]:
# Cell 2 — Helper functions

def clean_text(t: str) -> str:
    if not isinstance(t, str):
        return ""
    t = t.replace("\r", " ").replace("\n", " ").strip()
    t = re.sub(r"\s+", " ", t)
    return t

def first_sentence_as_title(txt: str, max_len=120) -> str:
    if not txt:
        return ""
    m = re.search(r"(.+?\.)\s", txt)
    cand = m.group(1) if m else txt[:max_len]
    return clean_text(cand)[:max_len]

def parse_domain(u: str) -> str:
    if not isinstance(u, str) or not u:
        return ""
    try:
        return urlparse(u).netloc.lower()
    except Exception:
        return ""

def normalize_date_iso(dstr: str) -> str:
    """
    Return YYYY-MM-DD or '' if not parseable.
    MLSUM can have '00/06/2019' -> set day to 1 -> '2019-06-01'.
    """
    if not isinstance(dstr, str) or not dstr.strip():
        return ""
    s = dstr.strip()

    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%Y.%m.%d"):
        try:
            return datetime.strptime(s, fmt).strftime("%Y-%m-%d")
        except:
            pass

    mm = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", s)
    if mm:
        d, m, y = int(mm.group(1)), int(mm.group(2)), int(mm.group(3))
        d = 1 if d == 0 else d
        try:
            return datetime(year=y, month=m, day=d).strftime("%Y-%m-%d")
        except:
            return ""
    return ""

def sample_df(df: pd.DataFrame, n: int) -> pd.DataFrame:
    if len(df) <= n:
        return df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    return df.sample(n=n, random_state=42).reset_index(drop=True)

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df


# Inspect CNN/DailyMail (CSV)

In [27]:
# Cell 3 — Inspect inputs

from pprint import pprint

print("CNN/DailyMail files:")
pprint(sorted([str(p) for p in P_CNNDM.glob("*.csv")]))

print("\nXL-Sum files (first 20):")
pprint(sorted([p.name for p in P_XLSUM.glob("*.jsonl")])[:20])

print("\nMLSUM files (first 20):")
pprint(sorted([p.name for p in P_MLSUM.glob("*.csv")])[:20])


CNN/DailyMail files:
['/kaggle/input/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/test.csv',
 '/kaggle/input/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/train.csv',
 '/kaggle/input/newspaper-text-summarization-cnn-dailymail/cnn_dailymail/validation.csv']

XL-Sum files (first 20):
['amharic_test.jsonl',
 'amharic_train.jsonl',
 'amharic_val.jsonl',
 'arabic_test.jsonl',
 'arabic_train.jsonl',
 'arabic_val.jsonl',
 'azerbaijani_test.jsonl',
 'azerbaijani_train.jsonl',
 'azerbaijani_val.jsonl',
 'bengali_test.jsonl',
 'bengali_train.jsonl',
 'bengali_val.jsonl',
 'burmese_test.jsonl',
 'burmese_train.jsonl',
 'burmese_val.jsonl',
 'chinese_simplified_test.jsonl',
 'chinese_simplified_train.jsonl',
 'chinese_simplified_val.jsonl',
 'chinese_traditional_test.jsonl',
 'chinese_traditional_train.jsonl']

MLSUM files (first 20):
['de_test.csv',
 'de_train.csv',
 'de_validation.csv',
 'es_test.csv',
 'es_train.csv',
 'es_validation.csv',
 'fr_test.csv',
 'fr_train.csv',

# Sample CNN/DailyMail and tag language

In [28]:
# PURPOSE: Load and preview CNN/DailyMail CSV splits.

cnndm_paths = [P_CNNDM/"train.csv", P_CNNDM/"validation.csv", P_CNNDM/"test.csv"]
usecols_cd = ["id", "article", "highlights"]

cnndm_parts = []
for p in cnndm_paths:
    if p.exists():
        print("Found:", p.name)
        df = pd.read_csv(p, usecols=usecols_cd)
        print("  shape:", df.shape)
        display(df.head(3))
        cnndm_parts.append(df)

cnndm = pd.concat(cnndm_parts, axis=0, ignore_index=True)
print("CNN/DM combined shape:", cnndm.shape)


Found: train.csv
  shape: (287113, 3)


,id,article,highlights
0,0001d1afc246a7964130f43ae940af6bc6c57f01,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,0002095e55fcbd3a2f366d9bf92a95433dc305ef,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,00027e965c8264c35cc1bc55556db388da82b07f,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."


Found: validation.csv
  shape: (13368, 3)


,id,article,highlights
0,61df4979ac5fcc2b71be46ed6fe5a46ce7f071c3,"Sally Forrest, an actress-dancer who graced th...","Sally Forrest, an actress-dancer who graced th..."
1,21c0bd69b7e7df285c3d1b1cf56d4da925980a68,A middle-school teacher in China has inked hun...,Works include pictures of Presidential Palace ...
2,56f340189cd128194b2e7cb8c26bb900e3a848b4,A man convicted of killing the father and sist...,"Iftekhar Murtaza, 29, was convicted a year ago..."


Found: test.csv
  shape: (11490, 3)


,id,article,highlights
0,92c514c913c0bdfe25341af9fd72b29db544099b,Ever noticed how plane seats appear to be gett...,Experts question if packed out planes are put...
1,2003841c7dc0e7c5b1a248f9cd536d727f27a45a,A drunk teenage boy had to be rescued by secur...,Drunk teenage boy climbed into lion enclosure ...
2,91b7d2311527f5c2b63a65ca98d21d9c92485149,Dougie Freedman is on the verge of agreeing a ...,Nottingham Forest are close to extending Dougi...


CNN/DM combined shape: (311971, 3)


In [29]:
# PURPOSE: Downsample CNN/DM for tonight, add language column.

cnndm_small = sample_df(cnndm, N_CNNDM)
cnndm_small["language"] = "en"

print("CNN/DM sampled shape:", cnndm_small.shape)
display(cnndm_small.head(3))


CNN/DM sampled shape: (3000, 4)


,id,article,highlights,language
0,a1952ce5bce7d89d6d5a8b3efe647d485b3d3ebb,(CNN) -- Artist Rachel Sussman spent the past ...,"Artist Rachel Sussman's book, ""The Oldest Livi...",en
1,a312541b75fddacc2abb30583f62875a8565f3f7,A Turkish court has ordered authorities to blo...,Turkish court orders Facebook pages insulting ...,en
2,3ce5218f0593a4376bf61041f5dc4e6de1387a36,US Marine Corps Col. George Bristol commanded ...,Col. George Bristol was in charge of Special O...,en


# Inspect XL-Sum (JSONL) file list

In [30]:
# PURPOSE: List a few candidate XL-Sum files to confirm availability.

from itertools import islice

for lang_code, stem in XLSUM_LANGS.items():
    print(f"\n--- XL-Sum language: {lang_code} ({stem}) ---")
    for split in ["val", "validation", "test", "train"]:
        fp = P_XLSUM/f"{stem}_{split}.jsonl"
        print("  ", fp.name, "exists?" , fp.exists())



--- XL-Sum language: en (english) ---
   english_val.jsonl exists? True
   english_validation.jsonl exists? False
   english_test.jsonl exists? True
   english_train.jsonl exists? True

--- XL-Sum language: fr (french) ---
   french_val.jsonl exists? True
   french_validation.jsonl exists? False
   french_test.jsonl exists? True
   french_train.jsonl exists? True


# Load & preview XL-Sum (per language, one split)

In [32]:
# PURPOSE: For each selected language, load the first available split (val/test/train),
#          preview a few rows, then sample per_lang items.

xlsum_parts = []
per_lang = max(1, N_XLSUM // max(1, len(XLSUM_LANGS)))

for lang_code, stem in XLSUM_LANGS.items():
    loaded = False
    for split in ["val", "validation", "test", "train"]:
        fp = P_XLSUM/f"{stem}_{split}.jsonl"
        if fp.exists():
            print(f"\nLoading XL-Sum: {fp.name} for lang={lang_code}")
            rows = []
            with open(fp, "r", encoding="utf-8") as f:
                for i, line in enumerate(f):
                    try:
                        j = json.loads(line)
                        rows.append(j)
                    except:
                        continue
                    if len(rows) >= (per_lang * 3):  # read some headroom then sample
                        break
            dfx = pd.DataFrame(rows)
            # Ensure typical cols exist
            for c in ["text","summary","url","title"]:
                if c not in dfx.columns:
                    dfx[c] = ""
            dfx["language"] = lang_code
            print("  raw shape:", dfx.shape)
            display(dfx.head(3)[["title","text","summary","url"]])

            dfx_small = sample_df(dfx, per_lang)
            xlsum_parts.append(dfx_small)
            print("  sampled shape:", dfx_small.shape)
            loaded = True
            break
    if not loaded:
        print(f"WARNING: No XL-Sum split found for {lang_code}")

xlsum = pd.concat(xlsum_parts, axis=0, ignore_index=True) if xlsum_parts else pd.DataFrame(columns=["text","summary","url","title","language"])
print("\nXL-Sum combined sampled shape:", xlsum.shape)
display(xlsum.head(5))




Loading XL-Sum: english_val.jsonl for lang=en
  raw shape: (4500, 6)


,title,text,summary,url
0,Portsmouth sea wall hole repair work delayed,A large void was temporarily plugged with gran...,Work to repair a sea wall that collapsed durin...,https://www.bbc.com/news/uk-england-hampshire-...
1,What's happening in Bolivia?,His resignation on Sunday sparked clashes in c...,Bolivia's President Evo Morales has now left t...,https://www.bbc.com/news/newsbeat-50388934
2,Two fifths of NI electricity from renewables,By Conor MacauleyBBC NI Agriculture & Environm...,Almost 40% of the electricity generated in NI ...,https://www.bbc.com/news/uk-northern-ireland-4...


  sampled shape: (1500, 6)

Loading XL-Sum: french_val.jsonl for lang=fr
  raw shape: (1086, 6)


,title,text,summary,url
0,Des scientifiques africains cherchent à cartog...,L'initiative vise à étudier les éventuelles mu...,L'Organisation mondiale de la santé (OMS) et l...,https://www.bbc.com/afrique/monde-54118749
1,Trois braconniers nigérians écroués au Niger,Le Parc du W abrite une riche faune Leur équip...,Visiblement bien armés et ayant une certaine m...,https://www.bbc.com/afrique/region-47609666
2,Le crash d'un hélicoptère fait 18 morts en Sib...,Ces types d'hélicoptères assurent le transport...,Les 3 membres d'équipage et les 15 travailleur...,https://www.bbc.com/afrique/monde-45069427


  sampled shape: (1086, 6)

XL-Sum combined sampled shape: (2586, 6)


,id,url,title,summary,text,language
0,uk-wales-politics-35708741,https://www.bbc.co.uk/news/uk-wales-politics-3...,Ministers 'blocking higher Chinese steel tariffs',"UK ministers were among the ""ringleaders"" bloc...",UK Steel policy chief Dominic King also accuse...,en
1,uk-37098751,https://www.bbc.com/news/uk-37098751,Radical cleric Anjem Choudary guilty of inviti...,One of the UK's most notorious radical clerics...,"Police said Anjem Choudary, 49, had stayed ""ju...",en
2,newsbeat-32380304,https://www.bbc.com/news/newsbeat-32380304,Benedict Cumberbatch's Sherlock goes back to 1895,Benedict Cumberbatch's Sherlock and Martin Fre...,By Natalie JamiesonNewsbeat entertainment repo...,en
3,uk-england-bristol-44411263,https://www.bbc.com/news/uk-england-bristol-44...,Giant diesel soot particle unveiled in Bristol,A 3m (10ft) high diesel soot particle has been...,"The sculpture, which is three million times th...",en
4,entertainment-arts-31113633,https://www.bbc.com/news/entertainment-arts-31...,Taking The Casual Vacancy from the page to the...,The writer and director of the BBC One dramati...,By Neil SmithArts reporter Eyebrows were raise...,en


# Inspect MLSUM-FR (CSV)

In [33]:
# PURPOSE: Load and preview MLSUM French splits.

ml_fr_paths = [P_MLSUM/"fr_validation.csv", P_MLSUM/"fr_test.csv", P_MLSUM/"fr_train.csv"]
usecols_ml = ["text","summary","topic","url","title","date"]

ml_fr_parts = []
for p in ml_fr_paths:
    if p.exists():
        print("Found:", p.name)
        # Read header to guard missing columns (some mirrors differ)
        cols_available = pd.read_csv(p, nrows=0).columns.tolist()
        use = [c for c in usecols_ml if c in cols_available]
        df = pd.read_csv(p, usecols=use)
        print("  shape:", df.shape)
        display(df.head(3))
        ml_fr_parts.append(df)

mlsum_fr = pd.concat(ml_fr_parts, axis=0, ignore_index=True) if ml_fr_parts else pd.DataFrame(columns=usecols_ml)
mlsum_fr["language"] = "fr"
print("MLSUM-FR combined shape:", mlsum_fr.shape)


Found: fr_validation.csv
  shape: (16059, 6)


,text,summary,topic,url,title,date
0,Le Mali a entamé l’année 2019 avec une attaque...,"Les violences intercommunautaires, mêlées aux ...",afrique,https://www.lemonde.fr/afrique/article/2019/01...,Nouvel An meurtrier au Mali avec l’attaque d’u...,01/01/2019
1,C’était une condition pour suspendre les sanct...,"Accusé en 2016 de dopage d’Etat, le pays devai...",sport,https://www.lemonde.fr/sport/article/2019/01/0...,La Russie n’a pas transmis les données de son ...,01/01/2019
2,Une vingtaine de formations politiques soudana...,Le pays est en proie à des « émeutes du pain »...,afrique,https://www.lemonde.fr/afrique/article/2019/01...,"Au Soudan, une vingtaine de formations politiq...",01/01/2019


Found: fr_test.csv
  shape: (15828, 6)


,text,summary,topic,url,title,date
0,Le défenseur de Liverpool Virgil van Dijk célè...,Mohamed Salah et Divock Origi ont permis à Liv...,football,https://www.lemonde.fr/football/article/2019/0...,Liverpool remporte sa sixième Ligue des champi...,01/06/2019
1,"Michel Serres, en mai 2012. MANUEL COHEN Parmi...","Le philosophe, mort samedi à 88 ans, a su fair...",culture,https://www.lemonde.fr/culture/article/2019/06...,"Michel Serres, une forme française d’universalité",01/06/2019
2,"A Vincennes, en septembre 2018. Serge Picard /...","L’auteur des best-sellers « Les Cinq Sens », «...",disparitions,https://www.lemonde.fr/disparitions/article/20...,Le philosophe et académicien Michel Serres est...,01/06/2019


Found: fr_train.csv
  shape: (392902, 6)


,text,summary,topic,url,title,date
0,"Jean-Jacques Schuhl, Gilles Leroy, Christian G...","Jean-Jacques Schuhl, Gilles Leroy, Christian G...",livres,https://www.lemonde.fr/livres/article/2010/01/...,La rentrée littéraire promet un programme de b...,01/01/2010
1,Une semaine après l'attaque terroriste manquée...,Cette demande intervient une semaine après l'a...,proche-orient,https://www.lemonde.fr/proche-orient/article/2...,Gordon Brown appelle à une réunion internation...,01/01/2010
2,"Un juge américain a rejeté, jeudi 31 décembre,...",Un juge américain a rejeté jeudi les accusatio...,asie-pacifique,https://www.lemonde.fr/asie-pacifique/article/...,L'abandon des poursuites contre Blackwater ind...,01/01/2010


MLSUM-FR combined shape: (424789, 7)


# Sample MLSUM-FR

In [34]:
# PURPOSE: Downsample MLSUM-FR for tonight.

mlsum_fr_small = sample_df(mlsum_fr, N_MLSUMFR)
print("MLSUM-FR sampled shape:", mlsum_fr_small.shape)
display(mlsum_fr_small.head(3))


MLSUM-FR sampled shape: (3000, 7)


,text,summary,topic,url,title,date,language
0,La grande pyramide de Khéops sur le plateau de...,"Tourisme en berne, exportations en nette décél...",afrique,https://www.lemonde.fr/afrique/article/2011/05...,L'économie égyptienne est dans un état critique,25/05/2011,fr
1,Les révoltes arabes nées en Tunisie fin 2010 o...,"""Printemps"" ou pas, elle reste la figure de pr...",culture,https://www.lemonde.fr/culture/article/2012/03...,Oum Kalsoum règne encore sur la Méditerranée,17/03/2012,fr
2,"La sélection de Fabienne Darge « Joueurs, Mao ...","Théâtre, danse, opéra, humour : les critiques ...",culture,https://www.lemonde.fr/culture/article/2018/12...,Tous nos spectacles préférés de 2018,27/12/2018,fr


# Normalize each dataset to a unified schema

In [36]:
# PURPOSE: Map to a single schema the ranking/summarization code will expect.

def norm_cnndm(df):
    return pd.DataFrame({
        "id": df["id"].astype(str),
        "title": df["article"].apply(first_sentence_as_title),
        "text": df["article"].fillna("").apply(clean_text),
        "summary_ref": df["highlights"].fillna("").apply(clean_text),
        "url": "",
        "source": "cnn_dailymail",
        "date": "",
        "language": "en",
        "topic": ""
    })

def norm_xlsum(df):
    df = ensure_cols(df, ["text","summary","url","title","language"])
    df = df.copy()
    df["__rowid__"] = np.arange(len(df)).astype(str)
    src_domains = df["url"].fillna("").apply(parse_domain)
    # Fallback source if URL missing → avoid empty strings in Focus prior
    src_domains = np.where(src_domains == "", "xlsum_" + df["language"].astype(str), src_domains)
    return pd.DataFrame({
        "id": "xlsum_" + df["language"].astype(str) + "_" + df["__rowid__"],
        "title": np.where(df["title"].fillna("")!="", df["title"].fillna(""),
                          df["text"].fillna("").apply(first_sentence_as_title)),
        "text": df["text"].fillna("").apply(clean_text),
        "summary_ref": df["summary"].fillna("").apply(clean_text),
        "url": df["url"].fillna(""),
        "source": src_domains,
        "date": "",  # most mirrors don’t include dates
        "language": df["language"],
        "topic": ""
    })


def norm_mlsum_fr(df):
    df = ensure_cols(df, ["text","summary","topic","url","title","date"])
    ids = [f"mlsum_fr_{i}" for i in range(len(df))]
    return pd.DataFrame({
        "id": ids,
        "title": np.where(df["title"].fillna("")!="", df["title"].fillna(""),
                          df["text"].fillna("").apply(first_sentence_as_title)),
        "text": df["text"].fillna("").apply(clean_text),
        "summary_ref": df["summary"].fillna("").apply(clean_text),
        "url": df["url"].fillna(""),
        "source": df["url"].fillna("").apply(parse_domain).replace("", "mlsum_fr"),
        "date": df["date"].fillna("").apply(normalize_date_iso),
        "language": "fr",
        "topic": df["topic"].fillna("")
    })


cnndm_norm    = norm_cnndm(cnndm_small)
xlsum_norm    = norm_xlsum(xlsum)
mlsum_fr_norm = norm_mlsum_fr(mlsum_fr_small)

print("Shapes:", cnndm_norm.shape, xlsum_norm.shape, mlsum_fr_norm.shape)
display(cnndm_norm.head(2))
display(xlsum_norm.head(2))
display(mlsum_fr_norm.head(2))


Shapes: (3000, 9) (2586, 9) (3000, 9)


,id,title,text,summary_ref,url,source,date,language,topic
0,a1952ce5bce7d89d6d5a8b3efe647d485b3d3ebb,(CNN) -- Artist Rachel Sussman spent the past ...,(CNN) -- Artist Rachel Sussman spent the past ...,"Artist Rachel Sussman's book, ""The Oldest Livi...",,cnn_dailymail,,en,
1,a312541b75fddacc2abb30583f62875a8565f3f7,A Turkish court has ordered authorities to blo...,A Turkish court has ordered authorities to blo...,Turkish court orders Facebook pages insulting ...,,cnn_dailymail,,en,


,id,title,text,summary_ref,url,source,date,language,topic
0,xlsum_en_0,Ministers 'blocking higher Chinese steel tariffs',UK Steel policy chief Dominic King also accuse...,"UK ministers were among the ""ringleaders"" bloc...",https://www.bbc.co.uk/news/uk-wales-politics-3...,www.bbc.co.uk,,en,
1,xlsum_en_1,Radical cleric Anjem Choudary guilty of inviti...,"Police said Anjem Choudary, 49, had stayed ""ju...",One of the UK's most notorious radical clerics...,https://www.bbc.com/news/uk-37098751,www.bbc.com,,en,


,id,title,text,summary_ref,url,source,date,language,topic
0,mlsum_fr_0,L'économie égyptienne est dans un état critique,La grande pyramide de Khéops sur le plateau de...,"Tourisme en berne, exportations en nette décél...",https://www.lemonde.fr/afrique/article/2011/05...,www.lemonde.fr,2011-05-25,fr,afrique
1,mlsum_fr_1,Oum Kalsoum règne encore sur la Méditerranée,Les révoltes arabes nées en Tunisie fin 2010 o...,"""Printemps"" ou pas, elle reste la figure de pr...",https://www.lemonde.fr/culture/article/2012/03...,www.lemonde.fr,2012-03-17,fr,culture


# Concatenate, clean, filter, de-duplicate

In [37]:
# PURPOSE: Merge and enforce minimal quality filters.

all_df = pd.concat([cnndm_norm, xlsum_norm, mlsum_fr_norm], axis=0, ignore_index=True)

# Text length filter keeps things summarizable and not empty
all_df["text_len"] = all_df["text"].str.len()
all_df = all_df[all_df["text_len"].between(200, 10000)]

# Keep only EN/FR
all_df = all_df[all_df["language"].isin(KEEP_LANGS)]

# Ensure titles exist
mask_empty_title = all_df["title"].fillna("") == ""
all_df.loc[mask_empty_title, "title"] = all_df.loc[mask_empty_title, "text"].apply(first_sentence_as_title)

# Drop dupes
before = len(all_df)
all_df = all_df.drop_duplicates(subset=["title","text"])
after_1 = len(all_df)
if "url" in all_df.columns:
    all_df = all_df.drop_duplicates(subset=["url"], keep="first")
after_2 = len(all_df)

print(f"Rows before: {before} | after (title,text): {after_1} | after URL: {after_2}")
display(all_df.head(5)[["id","language","title","source","date"]])


Rows before: 8526 | after (title,text): 8525 | after URL: 5545


,id,language,title,source,date
0,a1952ce5bce7d89d6d5a8b3efe647d485b3d3ebb,en,(CNN) -- Artist Rachel Sussman spent the past ...,cnn_dailymail,
3000,xlsum_en_0,en,Ministers 'blocking higher Chinese steel tariffs',www.bbc.co.uk,
3001,xlsum_en_1,en,Radical cleric Anjem Choudary guilty of inviti...,www.bbc.com,
3002,xlsum_en_2,en,Benedict Cumberbatch's Sherlock goes back to 1895,www.bbc.com,
3003,xlsum_en_3,en,Giant diesel soot particle unveiled in Bristol,www.bbc.com,


In [38]:
display(all_df.head(5))


,id,title,text,summary_ref,url,source,date,language,topic,text_len
0,a1952ce5bce7d89d6d5a8b3efe647d485b3d3ebb,(CNN) -- Artist Rachel Sussman spent the past ...,(CNN) -- Artist Rachel Sussman spent the past ...,"Artist Rachel Sussman's book, ""The Oldest Livi...",,cnn_dailymail,,en,,3876
3000,xlsum_en_0,Ministers 'blocking higher Chinese steel tariffs',UK Steel policy chief Dominic King also accuse...,"UK ministers were among the ""ringleaders"" bloc...",https://www.bbc.co.uk/news/uk-wales-politics-3...,www.bbc.co.uk,,en,,3164
3001,xlsum_en_1,Radical cleric Anjem Choudary guilty of inviti...,"Police said Anjem Choudary, 49, had stayed ""ju...",One of the UK's most notorious radical clerics...,https://www.bbc.com/news/uk-37098751,www.bbc.com,,en,,6079
3002,xlsum_en_2,Benedict Cumberbatch's Sherlock goes back to 1895,By Natalie JamiesonNewsbeat entertainment repo...,Benedict Cumberbatch's Sherlock and Martin Fre...,https://www.bbc.com/news/newsbeat-32380304,www.bbc.com,,en,,1532
3003,xlsum_en_3,Giant diesel soot particle unveiled in Bristol,"The sculpture, which is three million times th...",A 3m (10ft) high diesel soot particle has been...,https://www.bbc.com/news/uk-england-bristol-44...,www.bbc.com,,en,,1387


- Confirm schema, preview, and inspect the longest-text row (fixed)

In [39]:
# PURPOSE: Confirm schema, preview, and inspect the longest-text row (fixed)

print("All columns:", list(all_df.columns))
print("Shape:", all_df.shape)

cols_to_peek = ["id","language","title","text","summary_ref","url","source","date","topic"]
display(all_df[cols_to_peek].head(2))

# Longest text row — OPTION 1: by label (uses current index labels)
longest_label = all_df["text"].str.len().idxmax()
row_i = all_df.loc[longest_label]

print("\nLongest text example:")
print("ID:", row_i["id"])
print("Title:", row_i["title"])
print("Source:", row_i["source"], "| Date:", row_i["date"])
print("Text chars:", len(row_i["text"]), "| Summary_ref chars:", len(row_i["summary_ref"]))
print("URL:", row_i["url"])


All columns: ['id', 'title', 'text', 'summary_ref', 'url', 'source', 'date', 'language', 'topic', 'text_len']
Shape: (5545, 10)


,id,language,title,text,summary_ref,url,source,date,topic
0,a1952ce5bce7d89d6d5a8b3efe647d485b3d3ebb,en,(CNN) -- Artist Rachel Sussman spent the past ...,(CNN) -- Artist Rachel Sussman spent the past ...,"Artist Rachel Sussman's book, ""The Oldest Livi...",,cnn_dailymail,,
3000,xlsum_en_0,en,Ministers 'blocking higher Chinese steel tariffs',UK Steel policy chief Dominic King also accuse...,"UK ministers were among the ""ringleaders"" bloc...",https://www.bbc.co.uk/news/uk-wales-politics-3...,www.bbc.co.uk,,



Longest text example:
ID: mlsum_fr_1769
Title: La Weinstein Company contrainte de se vendre
Source: www.lemonde.fr | Date: 2017-10-17
Text chars: 9946 | Summary_ref chars: 132
URL: https://www.lemonde.fr/economie/article/2017/10/17/la-weinstein-company-contrainte-de-se-vendre_5201929_3234.html


# Credibility prior (build + save)

This converts source domains to a small numeric prior the ranker can use.

In [40]:
# PURPOSE: Build and persist a tiny, transparent credibility prior.

top_k = 15                       # score only the top N domains
hi, lo, step = 0.90, 0.40, 0.02  # 0.90 -> 0.40 in steps of 0.02

src_counts = all_df["source"].fillna("").replace("", "unknown").value_counts()
top_src = [s for s in src_counts.index if s != "unknown"][:top_k]

cred_prior = {}
for i, s in enumerate(top_src):
    cred_prior[s] = round(max(lo, hi - i*step), 2)

DEFAULT_CRED = 0.50

print("Top domains:")
display(src_counts.head(10))
print("\nCred prior preview:", list(cred_prior.items())[:8], " | default:", DEFAULT_CRED)

# Save to disk for later cells / reuse
import json
OUT_CRED = "/kaggle/working/credibility_prior.json"
with open(OUT_CRED, "w", encoding="utf-8") as f:
    json.dump({"prior": cred_prior, "default": DEFAULT_CRED}, f, ensure_ascii=False, indent=2)
print("Saved:", OUT_CRED)


Top domains:


source
www.lemonde.fr                       2921
www.bbc.com                          2535
www.bbc.co.uk                          51
sosconso.blog.lemonde.fr                4
autourduciel.blog.lemonde.fr            3
tourdefrance.blog.lemonde.fr            2
voileactu.blog.lemonde.fr               2
passeurdesciences.blog.lemonde.fr       2
bigbrowser.blog.lemonde.fr              2
ecosport.blog.lemonde.fr                2
Name: count, dtype: int64


Cred prior preview: [('www.lemonde.fr', 0.9), ('www.bbc.com', 0.88), ('www.bbc.co.uk', 0.86), ('sosconso.blog.lemonde.fr', 0.84), ('autourduciel.blog.lemonde.fr', 0.82), ('tourdefrance.blog.lemonde.fr', 0.8), ('voileactu.blog.lemonde.fr', 0.78), ('passeurdesciences.blog.lemonde.fr', 0.76)]  | default: 0.5
Saved: /kaggle/working/credibility_prior.json


# Export demo & eval artifacts

This freezes the cleaned data so the modeling section is reproducible.

In [41]:
# PURPOSE: Persist the cleaned corpus (for ranking/summarization) and eval pairs (for ROUGE later).

DEMO_COLS = ["id","title","text","summary_ref","url","source","date","language"]
corpus_for_demo = all_df[DEMO_COLS].copy()

EVAL_COLS = ["id","language","text","summary_ref","title","source"]
eval_pairs = all_df[EVAL_COLS].copy()
eval_pairs_small = eval_pairs.sample(n=min(500, len(eval_pairs)), random_state=42)

OUT_CORPUS = "/kaggle/working/corpus_for_demo.json"
OUT_EVAL   = "/kaggle/working/eval_pairs.json"

corpus_for_demo.to_json(OUT_CORPUS, orient="records", force_ascii=False, indent=2)
eval_pairs_small.to_json(OUT_EVAL, orient="records", force_ascii=False, indent=2)

print("Saved:")
print(" ", OUT_CORPUS)
print(" ", OUT_EVAL)


Saved:
  /kaggle/working/corpus_for_demo.json
  /kaggle/working/eval_pairs.json


# Embeddings & Index (with safe fallback)

The ranker needs relevance. We’ll try sentence embeddings; if the model can’t be downloaded on Kaggle, we fallback to TF-IDF (still works).

In [5]:
# PURPOSE: Create vector representations so we can measure relevance (= cosine similarity).

import numpy as np
import pandas as pd

# Load the frozen corpus
import json
with open(OUT_CORPUS, "r", encoding="utf-8") as f:
    corpus = json.load(f)
C = pd.DataFrame(corpus)

# Text we embed = title + newline + text (good tradeoff)
C["combined"] = C["title"].fillna("") + "\n" + C["text"].fillna("")

# Try sentence-transformers (multilingual); otherwise fallback to TF-IDF.
use_st = True
embeddings = None
embed_explainer = ""

try:
    from sentence_transformers import SentenceTransformer
    st_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # EN + FR
    st_model = SentenceTransformer(st_model_name)
    X = st_model.encode(C["combined"].tolist(), normalize_embeddings=True, show_progress_bar=False)
    X = np.asarray(X, dtype="float32")
    embed_explainer = f"Embeddings: {st_model_name} (normalized)"
except Exception as e:
    use_st = False
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize
    print("Sentence-Transformers unavailable -> falling back to TF-IDF.")
    tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
    X = tfidf.fit_transform(C["combined"].tolist())
    X = normalize(X, norm="l2", copy=False)  # cosine-compatible
    embed_explainer = "Embeddings: TF-IDF (L2-normalized vectors)"

print(embed_explainer)
print("Vector shape:", X.shape)


2025-10-27 13:15:24.399187: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761570924.554205      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761570924.600323      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

Sentence-Transformers unavailable -> falling back to TF-IDF.
Embeddings: TF-IDF (L2-normalized vectors)
Vector shape: (3974, 50000)


# Recency function (exponential decay)

We convert dates into a freshness number in 0,1. Missing date → use 0.5 (neutral).

In [6]:
# PURPOSE: Map dates to a freshness score using exponential decay.

from datetime import datetime

# If your demo should act "as of today", set 'today' once.
# To keep it deterministic across runs, pick a fixed date or parse max date in corpus.
today = datetime(2020, 12, 31)  # <-- adjust if you want; MLSUM is older. Fixed keeps scores stable.

def recency_score(date_str: str, tau_days: int = 4) -> float:
    """
    Exponential decay by days old: r = exp(-D / tau).
    If date missing, return 0.5 (neutral).
    """
    if not isinstance(date_str, str) or not date_str:
        return 0.5
    try:
        dt = datetime.strptime(date_str, "%Y-%m-%d")
        D = max(0, (today - dt).days)
        return float(np.exp(-D / float(tau_days)))
    except:
        return 0.5


# Scoring + Focus 3

This is the heart: we compute three numbers per article and blend them with weights.

In [7]:
# PURPOSE: Compute relevance, recency, credibility for a query, then a final score.

# Load credibility prior (built earlier)
with open("/kaggle/working/credibility_prior.json", "r", encoding="utf-8") as f:
    cred_blob = json.load(f)
CRED_PRIOR = cred_blob["prior"]
CRED_DEFAULT = cred_blob["default"]

# Weights (human-chosen for demo; later you can learn them)
ALPHA = 0.60  # relevance
BETA  = 0.30  # recency
GAMMA = 0.10  # credibility

# 1) Make a query embedding (or TF-IDF vector) to get relevance
def embed_query(q: str):
    if use_st:
        v = st_model.encode([q], normalize_embeddings=True)
        return np.asarray(v, dtype="float32")
    else:
        # TF-IDF fallback: vectorize with the same vocabulary; L2 norm already applied
        return tfidf.transform([q])

def cosine_topk(qv, X, k=30):
    if use_st:
        # Cosine == dot product because vectors are normalized
        sims = (X @ qv[0].reshape(-1,1)).ravel()
    else:
        sims = (X @ qv.T).toarray().ravel()
    idx = np.argsort(-sims)[:k]
    return sims, idx

# 2) Score function for a given query
def focus_score(query: str, k_candidates: int = 50):
    qv = embed_query(query)
    sims, idx = cosine_topk(qv, X, k=k_candidates)

    rows = []
    for rank, i in enumerate(idx):
        row = C.iloc[i]
        relevance = float(sims[i])
        recency   = recency_score(row["date"])
        credibility = float(CRED_PRIOR.get(row["source"], CRED_DEFAULT))
        score = ALPHA*relevance + BETA*recency + GAMMA*credibility

        rows.append({
            "rank_cand": rank+1,
            "id": row["id"],
            "language": row["language"],
            "title": row["title"][:160],
            "source": row["source"],
            "date": row["date"],
            "relevance": round(relevance, 4),
            "recency": round(recency, 4),
            "credibility": round(credibility, 2),
            "score": round(score, 4),
            "url": row["url"]
        })

    df_scores = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    return df_scores

# DEMO: try a query
demo_query = "AI in healthcare"  # change for your persona/topic
focus_table = focus_score(demo_query, k_candidates=80)
print("Query:", demo_query)
display(focus_table.head(10))

# Focus 3 (what you’ll show in the presentation)
focus3 = focus_table.head(3)
focus3


Query: AI in healthcare


,rank_cand,id,language,title,source,date,relevance,recency,credibility,score,url
0,1,xlsum_en_704,en,US healthcare 'to blame' for poor life expecta...,www.bbc.com,,0.2260,0.5,0.9,0.3756,https://www.bbc.com/news/world-us-canada-11502938
1,2,xlsum_en_565,en,US election 2020: Elizabeth Warren sets out he...,www.bbc.com,,0.1273,0.5,0.9,0.3164,https://www.bbc.com/news/world-us-canada-50267557
2,3,xlsum_en_524,en,NHS surgery waits run into years in Northern I...,www.bbc.com,,0.1264,0.5,0.9,0.3159,https://www.bbc.com/news/health-41649870
3,4,xlsum_fr_1159,fr,'J'étais la seule femme journaliste dans ma vi...,www.bbc.com,,0.1184,0.5,0.9,0.3110,https://www.bbc.com/afrique/region-49817387
4,6,xlsum_en_850,en,Clinical waste from Scotland to be sent to Wales,www.bbc.com,,0.1142,0.5,0.9,0.3085,https://www.bbc.com/news/uk-scotland-48298987
5,7,xlsum_en_520,en,Heatwave: Health warnings issued for south-eas...,www.bbc.com,,0.1105,0.5,0.9,0.3063,https://www.bbc.com/news/uk-23341504
6,9,xlsum_en_420,en,Nigeria kidnapping: Doctor says he has forgive...,www.bbc.com,,0.0971,0.5,0.9,0.2983,https://www.bbc.com/news/uk-england-cambridges...
7,10,xlsum_en_232,en,Third of NI students leave for universities el...,www.bbc.com,,0.0936,0.5,0.9,0.2961,https://www.bbc.com/news/uk-northern-ireland-4...
8,11,xlsum_en_851,en,Coronavirus: Bars to shut in four more French ...,www.bbc.com,,0.0933,0.5,0.9,0.2960,https://www.bbc.com/news/world-europe-54465174
9,12,xlsum_fr_1135,fr,Agyemang Badu : mon combat contre la mort et p...,www.bbc.com,,0.0915,0.5,0.9,0.2949,https://www.bbc.com/afrique/sports-52292789


,rank_cand,id,language,title,source,date,relevance,recency,credibility,score,url
0,1,xlsum_en_704,en,US healthcare 'to blame' for poor life expecta...,www.bbc.com,,0.2260,0.5,0.9,0.3756,https://www.bbc.com/news/world-us-canada-11502938
1,2,xlsum_en_565,en,US election 2020: Elizabeth Warren sets out he...,www.bbc.com,,0.1273,0.5,0.9,0.3164,https://www.bbc.com/news/world-us-canada-50267557
2,3,xlsum_en_524,en,NHS surgery waits run into years in Northern I...,www.bbc.com,,0.1264,0.5,0.9,0.3159,https://www.bbc.com/news/health-41649870


*How the model “understands the method of calculations”:* 

- Relevance: cosine between the query vector and each article vector.
- Recency: exp(-days/τ) from the ISO date string.
- Credibility: number looked up from your credibility_prior.json.
- Final score: a weighted sum of those three numbers (the “smoothie”).

Everything is numeric and visible in the focus_table columns—no black box.

# Summarizer (abstractive with fallback to extractive)

1) Try transformers (t5-small or distilbart); if not available,
2) Use a lightweight extractive fallback (TF-IDF sentence scoring).

In [8]:
# PURPOSE: Create a summarizer that ALWAYS works:
# 1) Try transformers (t5-small or distilbart); if not available,
# 2) Use a lightweight extractive fallback (TF-IDF sentence scoring).

from typing import List
import re

summarizer = None
sum_explainer = ""

def _split_sentences(txt: str) -> List[str]:
    # very light sentence splitter; good enough for demo
    sents = re.split(r'(?<=[.!?])\s+', txt.strip())
    # clean empties
    return [s.strip() for s in sents if len(s.strip()) > 0]

def _extractive_summary(text: str, max_sents: int = 2) -> str:
    # score sentences via TF-IDF against the whole doc; pick top-k
    sents = _split_sentences(text)
    if not sents:
        return ""
    use_k = min(max_sents, len(sents))
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        import numpy as np
        vect = TfidfVectorizer(max_features=8000, ngram_range=(1,2))
        X = vect.fit_transform(sents)  # sentence-term matrix
        # Sentence importance ~ L2 norm (or sum of tfidf weights)
        scores = (X.power(2).sum(axis=1)).A.ravel()
        top_idx = np.argsort(-scores)[:use_k]
        # keep original order for readability
        top_idx_sorted = sorted(top_idx.tolist())
        sel = [sents[i] for i in top_idx_sorted]
        return " ".join(sel)
    except Exception:
        # ultra fallback: first-k sentences (lead)
        return " ".join(sents[:use_k])

try:
    from transformers import pipeline
    # Prefer a small model to keep it snappy
    try:
        summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
        sum_explainer = "Summarizer: distilbart-cnn-12-6"
    except Exception:
        summarizer = pipeline("summarization", model="t5-small", tokenizer="t5-small")
        sum_explainer = "Summarizer: t5-small"
except Exception:
    summarizer = None
    sum_explainer = "Summarizer: extractive (TF-IDF fallback)"

print(sum_explainer)

def summarize_text(text: str, max_words: int = 40) -> str:
    """
    Return ~1–2 short sentences. We enforce brevity for the 'Digest' feel.
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    # Truncate very long inputs for safety
    trimmed = text.strip()
    if len(trimmed) > 4000:
        trimmed = trimmed[:4000]

    if summarizer is None or "extractive" in sum_explainer:
        # Extractive fallback path
        return _extractive_summary(trimmed, max_sents=2)

    # Abstractive path
    try:
        if "t5" in sum_explainer.lower():
            inp = "summarize: " + trimmed
            out = summarizer(inp, max_length=80, min_length=20, do_sample=False)[0]["summary_text"]
        else:
            out = summarizer(trimmed, max_length=80, min_length=20, do_sample=False)[0]["summary_text"]
        # limit to ~max_words words
        words = out.split()
        if len(words) > max_words:
            out = " ".join(words[:max_words]) + " …"
        return out
    except Exception:
        # If anything fails, use the extractive backup
        return _extractive_summary(trimmed, max_sents=2)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

Summarizer: extractive (TF-IDF fallback)


# Make the “Digest 5–10” from your Focus results

- PURPOSE: Build a digest: 5–10 concise items with bullets + link.
- We reuse focus_score() to get relevant candidates for the query.

In [9]:
# PURPOSE: Build a digest: 5–10 concise items with bullets + link.
# We reuse focus_score() to get relevant candidates for the query.

from IPython.display import Markdown, display

def make_digest(query: str, n_items: int = 8, k_candidates: int = 120):
    # Get candidates with our tri-feature score
    scores = focus_score(query, k_candidates=k_candidates)

    # Take the top n_items
    picks = scores.head(n_items).copy()

    # For each pick, generate 1–2 bullet points from the article text
    # (we look the text up from the corpus dataframe C by id)
    id_to_row = { row["id"]: row for _, row in C.iterrows() }

    md_lines = []
    md_lines.append(f"### Digest: {query}")
    md_lines.append(f"*({sum_explainer}; relevance α={ALPHA}, recency β={BETA}, credibility γ={GAMMA})*")
    md_lines.append("")

    for i, r in picks.iterrows():
        row = id_to_row.get(r["id"], None)
        text_for_sum = row["text"] if row is not None else ""
        short = summarize_text(text_for_sum, max_words=40)

        # Build a clickable title if URL exists
        title = r['title']
        if isinstance(r['url'], str) and len(r['url']) > 0:
            title_md = f"[{title}]({r['url']})"
        else:
            title_md = title

        md_lines.append(f"**{i+1}. {title_md}**  \n*{r['source']}*  {('(' + r['date'] + ')') if r['date'] else ''}")
        if short:
            # Turn into 1–2 bullets by splitting sentences
            bullets = re.split(r'(?<=[.!?])\s+', short.strip())
            bullets = [b for b in bullets if b]
            for b in bullets[:2]:
                md_lines.append(f"- {b}")
        md_lines.append("")

    display(Markdown("\n".join(md_lines)))
    return picks

# DEMO: generate a digest for a few topics
topics = [
    "AI in healthcare",
    "économie française",
    "football",
]

for t in topics:
    _ = make_digest(t, n_items=6, k_candidates=150)


### Digest: AI in healthcare
*(Summarizer: extractive (TF-IDF fallback); relevance α=0.6, recency β=0.3, credibility γ=0.1)*

**1. [US healthcare 'to blame' for poor life expectancy rates](https://www.bbc.com/news/world-us-canada-11502938)**  
*www.bbc.com*  
- Higher costs The study notes that in 1950, the US ranked fifth among leading industrialised nations for female life expectancy at birth, but only 46th in 2008.
- It finds that US healthcare spending increased at nearly twice the rate of that in other wealthy nations between 1970 and 2002, with the increased spending corresponding with worsening survival rates relative to the other countries studied.

**2. [US election 2020: Elizabeth Warren sets out health funding plan](https://www.bbc.com/news/world-us-canada-50267557)**  
*www.bbc.com*  
- She said "Medicare for All" would not raise taxes "one penny" for ordinary Americans, but would largely be paid for by businesses and the wealthy.
- But she said the share spent by the federal government would increase to $20.5tn.

**3. [NHS surgery waits run into years in Northern Ireland](https://www.bbc.com/news/health-41649870)**  
*www.bbc.com*  
- The BBC has obtained exclusive figures showing long waiting times before a decision to operate.
- Her mother Karen said: "Megan loves dancing.

**4. ['J'étais la seule femme journaliste dans ma ville'](https://www.bbc.com/afrique/region-49817387)**  
*www.bbc.com*  
- Maryan en direct lors d'une conférence de presse en Somalie Elle a raconté à la BBC son histoire relative aux défis auxquels les femmes sont confrontées dans l'un des pays les plus dangereux du monde pour devenir journaliste et comment elle a failli être tuée dans une attaque du groupe islamiste militant Al-Shabab.
- En Somalie, tes parents ne te demandent jamais ce que tu vas devenir à l'avenir, car en tant que femme, ta place est à la maison, à la cuisine et au ménage.

**5. [Clinical waste from Scotland to be sent to Wales](https://www.bbc.com/news/uk-scotland-48298987)**  
*www.bbc.com*  
- The new waste disposal contract for Scotland has been awarded to the Spanish-owned firm Tradebe Healthcare National and is due to commence in August.
- The Scottish government's health secretary, Jeane Freeman, gave details of the new disposal sites in answer to questions from the Conservative MSP for Central Scotland Graham Simpson.

**6. [Heatwave: Health warnings issued for south-east England](https://www.bbc.com/news/uk-23341504)**  
*www.bbc.com*  
- The night-time trigger temperatures vary from 15C to 18C.
- Scotland and Northern Ireland are not included in the alert system.


### Digest: économie française
*(Summarizer: extractive (TF-IDF fallback); relevance α=0.6, recency β=0.3, credibility γ=0.1)*

**1. [La note de l'Afrique du Sud dégradée](https://www.bbc.com/afrique/region-39491377)**  
*www.bbc.com*  
- L'agence de notation assure que l'incertitude politique et institutionnelle fait peser des risques importants sur l'économie.
- Lire aussi: Afrique du Sud : le Ministre des finances limogé Afrique du sud: Helen Zille sanctionnée pour ses tweets Cette dégradation de la note souveraine intervient après le remaniement ministériel de grande ampleur de la semaine.

**2. [Total accusée de fraude en Guinée Equatoriale](https://www.bbc.com/afrique/region-41091905)**  
*www.bbc.com*  
- Il précise que cette situation suspecte et frauduleuse a été décelée via ''des investigations gouvernementales''.
- Les responsables de la société française en Guinée équatoriale n'ont pas souhaité réagir a précisé l'AFP.

**3. [Un leader jihadiste tué au Mali](https://www.bbc.com/afrique/region-45329502)**  
*www.bbc.com*  
- Deux avions de type Mirage 2000 ont effectué les frappes L'un des leaders du groupe Etat islamique au Grand Sahara (EIGS) ainsi que deux civils ont été tués dans une opération de l'armée française dimanche dans la région de Ménaka.
- "Les commandos déployés (au sol après une frappe aérienne) ont constaté la mort de Mohamed Ag Almouner et d'un membre de sa garde rapprochée", selon un communiqué.

**4. [La colère contre d'anciens généraux qui mettent en garde contre une "guerre civile" en France](https://www.bbc.com/afrique/monde-56916215)**  
*www.bbc.com*  
- Les ministres ont condamné la missive publié dans un magazine d'extrême droite.
- "L'heure est grave, la France est en péril", ont déclaré les signataires.

**5. [Une Française chauffeur de taxi à Bamako](https://www.bbc.com/afrique/region/2016/03/160331_mali)**  
*www.bbc.com*  
- Ce sont des jeunes qui cherchaient un taxi pour lui.
- J’ai divisé le prix par deux, il a dit non.

**6. [Plainte du président Nkurunziza contre une chaine de Télévision française et 3 'complices'](https://www.bbc.com/afrique/region-46997905)**  
*www.bbc.com*  
- A lire aussi :Le Burundi suspend plusieurs ONG A lire aussi : Human Rights Watch dénonce la répression au Burundi "Au Burundi l'Etat cache les corps des personnes assassinées" L'émission montrait des massacres prétendument perpétrés par des partisans du président burundais à l'intérieur de l'une de ses propriétés.
- Le Président Nkurunziza a indiqué que les images n'ont pas été tournées au Burundi, mais quelque part en Afrique de l'Ouest.


### Digest: football
*(Summarizer: extractive (TF-IDF fallback); relevance α=0.6, recency β=0.3, credibility γ=0.1)*

**1. [Zeinab Camara, celle qui veut développer le football en Guinée](https://www.bbc.com/afrique/region-49168070)**  
*www.bbc.com*  
- La présidente du Fatala Football Club à Boffa, d'où elle est originaire, aide les plus jeunes, garçons et filles, à s'épanouir par le football.
- Zeinab Camara, présidente du Fatala Football Club à Boffa et cheffe de Cabinet du ministère de l'Enseignement supérieur guinéen.

**2. [Des mesures disciplinaires de la CAF contre Madagascar](https://www.bbc.com/afrique/sports-45692715)**  
*www.bbc.com*  
- Pendant trois mois, Madagascar n'accueillera pas de match international de football.
- La Fédération malgache de football est également condamnée à payer une amende de 10.000 dollars à cause de la bousculade.

**3. [Super Falcons : Florence Omagbemi non reconduite](https://www.bbc.com/afrique/sports-40988263)**  
*www.bbc.com*  
- L'entraineuse Florence Omagbemi a remporté avec les Super Falcons la dernière CAN féminine de football Le contrat d'Omagbemi a expiré à la fin de la triomphante campagne 2016 de la Coupe d'Afrique féminine de football au Cameroun.
- Florence Omagbeni en action lors de la Coupe du monde en 2003 Même si son nom figure sur la liste des 10 femmes nominées par la Fifa pour désigner l'entraîneuse de l'année, le nom d'Omagbemi n'apparaît pas sur la liste restreinte publiée par la Fédération Nigériane de Football (NFF) pour différents postes dans les équipes nationales.

**4. [Mantuidi et Moïse Kean, soutenus par la Fédération française de football](https://www.bbc.com/afrique/region-47802790)**  
*www.bbc.com*  
- Dans un communiqué, le président de la Fédération française de football, Noël Le Graëta a condamné les dérapages racistes dont ont été victimes Matuidi et Kean: "Ces insultes racistes sont inacceptables.
- h Lire aussi: Racisme : le plaidoyer de Sulley Muntari Arthur Masuaku: "les médias peuvent aider à lutter contre le racisme" Aubameyang victime de racisme Moise Kean comptabilise déjà deux sélections avec l'Italie.

**5. ["Le non-paiement de salaire m'a fait quitter le football pour la coiffure"](https://www.bbc.com/afrique/sports-53927951)**  
*www.bbc.com*  
- Le début de Bright Stars En 2019, Chinenye Okafor et Pelican Stars ont été immédiatement promus au premier rang du football féminin au Nigeria Okafor a rejoint les Stars en 2018 et bien que sa première saison avec le club se soit terminée par une relégation, elle a apprécié l'expérience et le style de vie.
- "Nous avons décidé qu'en tant que joueuses, puisque c'était nous qui avions mené l'équipe à la relégation, nous allions ramener l'équipe en Premier League." Elles ont tenu parole, et sont revenues avec une promotion immédiate qui a même été célébrée à la Maison du gouvernement de l'État de Cross River.

**6. [Football : un ancien entraîneur anglais reconnu coupable de pédophilie](https://www.bbc.com/afrique/sports-43053925)**  
*www.bbc.com*  
- ( Photo d'archive) C'est une affaire qui a entraîné une cascade de révélations de cas similaires et ébranlé le monde du football britannique.
- Bennell a été reconnu coupable d'attentats à la pudeur et d'actes de sodomie envers une dizaine de plaignants mardi.


# RAG definition

**What we implement here: Retrieval–Augmented Generation (RAG)**

- **Retrieve**: We embed each article (`title + text`) using either **TF-IDF** or **Sentence-Transformers** and retrieve top-K by **cosine similarity** to the user’s query.
- **Augment**: For each retrieved item we compute extra features:
  - **Recency**: `exp(-days_old / τ)` from the ISO date.
  - **Credibility**: a small, transparent **domain prior** built from our corpus (fallback default = 0.50).
- **Generate**: We produce 1–2 bullet summaries per item with an **abstractive model** (if available) or a robust **extractive TF-IDF** fallback.
- **Ranking for Focus 3**: final score = `α·relevance + β·recency + γ·credibility` (we use α=0.6, β=0.3, γ=0.1).  
We always show **source + link** for trust.

This keeps generation grounded in retrieved texts and explains how items are selected.


# Tools + renderer

In [1]:
# PURPOSE: RAG tools used by the agent. Reuses earlier cells/variables:
# C, focus_score(), summarize_text(), ALPHA/BETA/GAMMA

import re
import pandas as pd
from IPython.display import Markdown, display

def tool_focus(query: str, k_candidates: int = 120, top_n: int = 3) -> pd.DataFrame:
    """Focus Top-N with scores & links (RAG: retrieve -> rerank)."""
    return focus_score(query, k_candidates=k_candidates).head(top_n)

def tool_digest(query: str, n_items: int = 8, k_candidates: int = 180) -> pd.DataFrame:
    """Digest items with 1–2 bullets (RAG: retrieve -> rerank -> generate)."""
    scores = focus_score(query, k_candidates=k_candidates).head(n_items)
    id_to_row = { row["id"]: row for _, row in C.iterrows() }
    rows = []
    for _, r in scores.iterrows():
        base = id_to_row.get(r["id"])
        text = base["text"] if base is not None else ""
        bullet = summarize_text(text, max_words=40)
        rows.append({
            "title": r["title"],
            "source": r["source"],
            "date": r["date"],
            "url": r["url"],
            "bullet": bullet
        })
    return pd.DataFrame(rows)

def render_digest_markdown(df: pd.DataFrame, header: str) -> str:
    lines = [f"### {header}", f"*relevance α={ALPHA}, recency β={BETA}, credibility γ={GAMMA}*", ""]
    for i, row in df.iterrows():
        title_md = f"[{row['title']}]({row['url']})" if isinstance(row["url"], str) and row["url"] else row["title"]
        meta = f"*{row['source']}*"
        if row["date"]:
            meta += f" ({row['date']})"
        lines.append(f"**{i+1}. {title_md}**  \n{meta}")
        if row["bullet"]:
            bullets = re.split(r'(?<=[.!?])\s+', row["bullet"].strip())
            bullets = [b for b in bullets if b][:2]
            for b in bullets:
                lines.append(f"- {b}")
        lines.append("")
    return "\n".join(lines)


# The agent router + 3 demo calls

In [3]:
# PURPOSE: Minimal agent: routes 'focus:' and 'digest:' prompts to tools.

import re
from IPython.display import Markdown, display  # <-- import here, once

def news_agent(prompt: str):
    """
    Commands:
      - 'focus: <topic> [top=3]'
      - 'digest: <topic> [n=8]'
    If no prefix: returns both Focus(3) and Digest(6).
    """
    p = prompt.strip()
    low = p.lower()

    if low.startswith("focus:"):
        m = re.search(r"top\s*=\s*(\d+)", low)
        top_n = int(m.group(1)) if m else 3
        topic = p.split(":", 1)[1].strip()
        tbl = tool_focus(topic, top_n=top_n)
        display(tbl)
        return tbl

    if low.startswith("digest:"):
        m = re.search(r"[^\w]n\s*=\s*(\d+)", low)
        n = int(m.group(1)) if m else 8
        topic = p.split(":", 1)[1].strip()
        df = tool_digest(topic, n_items=n)
        display(Markdown(render_digest_markdown(df, header=f"Digest: {topic}")))
        return df

    # default: both
    tbl = tool_focus(p, top_n=3)
    df  = tool_digest(p, n_items=6)
    display(tbl)
    display(Markdown(render_digest_markdown(df, header=f"Digest: {p}")))
    return {"focus": tbl, "digest": df}

# DEMO (run these in Kaggle):
_ = news_agent("focus: AI in healthcare top=3")
_ = news_agent("digest: économie française n=6")
_ = news_agent("global headlines")


,rank_cand,id,language,title,source,date,relevance,recency,credibility,score,url
0,1,xlsum_en_704,en,US healthcare 'to blame' for poor life expecta...,www.bbc.com,,0.1934,0.5,0.9,0.3561,https://www.bbc.com/news/world-us-canada-11502938
1,2,xlsum_en_565,en,US election 2020: Elizabeth Warren sets out he...,www.bbc.com,,0.1090,0.5,0.9,0.3054,https://www.bbc.com/news/world-us-canada-50267557
2,3,xlsum_en_905,en,Qatari Sheikha tops 2013 art power list,www.bbc.com,,0.1087,0.5,0.9,0.3052,https://www.bbc.com/news/entertainment-arts-24...


### Digest: économie française n=6
*relevance α=0.6, recency β=0.3, credibility γ=0.1*

**1. [La note de l'Afrique du Sud dégradée](https://www.bbc.com/afrique/region-39491377)**  
*www.bbc.com*
- L'agence de notation assure que l'incertitude politique et institutionnelle fait peser des risques importants sur l'économie.
- Lire aussi: Afrique du Sud : le Ministre des finances limogé Afrique du sud: Helen Zille sanctionnée pour ses tweets Cette dégradation de la note souveraine intervient après le remaniement ministériel de grande ampleur de la semaine.

**2. [Total accusée de fraude en Guinée Equatoriale](https://www.bbc.com/afrique/region-41091905)**  
*www.bbc.com*
- Il précise que cette situation suspecte et frauduleuse a été décelée via ''des investigations gouvernementales''.
- Les responsables de la société française en Guinée équatoriale n'ont pas souhaité réagir a précisé l'AFP.

**3. [Un leader jihadiste tué au Mali](https://www.bbc.com/afrique/region-45329502)**  
*www.bbc.com*
- Deux avions de type Mirage 2000 ont effectué les frappes L'un des leaders du groupe Etat islamique au Grand Sahara (EIGS) ainsi que deux civils ont été tués dans une opération de l'armée française dimanche dans la région de Ménaka.
- "Les commandos déployés (au sol après une frappe aérienne) ont constaté la mort de Mohamed Ag Almouner et d'un membre de sa garde rapprochée", selon un communiqué.

**4. [La colère contre d'anciens généraux qui mettent en garde contre une "guerre civile" en France](https://www.bbc.com/afrique/monde-56916215)**  
*www.bbc.com*
- Les ministres ont condamné la missive publié dans un magazine d'extrême droite.
- "L'heure est grave, la France est en péril", ont déclaré les signataires.

**5. [Une Française chauffeur de taxi à Bamako](https://www.bbc.com/afrique/region/2016/03/160331_mali)**  
*www.bbc.com*
- Ce sont des jeunes qui cherchaient un taxi pour lui.
- J’ai divisé le prix par deux, il a dit non.

**6. [Plainte du président Nkurunziza contre une chaine de Télévision française et 3 'complices'](https://www.bbc.com/afrique/region-46997905)**  
*www.bbc.com*
- A lire aussi :Le Burundi suspend plusieurs ONG A lire aussi : Human Rights Watch dénonce la répression au Burundi "Au Burundi l'Etat cache les corps des personnes assassinées" L'émission montrait des massacres prétendument perpétrés par des partisans du président burundais à l'intérieur de l'une de ses propriétés.
- Le Président Nkurunziza a indiqué que les images n'ont pas été tournées au Burundi, mais quelque part en Afrique de l'Ouest.


,rank_cand,id,language,title,source,date,relevance,recency,credibility,score,url
0,1,xlsum_en_264,en,Aberdeen Asset Management reports new business...,www.bbc.com,,0.0766,0.5,0.9,0.2860,https://www.bbc.com/news/uk-scotland-scotland-...
1,2,xlsum_fr_1751,fr,Un accord pétrolier remis en cause au Nigeria,www.bbc.com,,0.0719,0.5,0.9,0.2831,https://www.bbc.com/afrique/region-46341856
2,3,xlsum_fr_1208,fr,"Environnement: ""l'humanité vit à crédit"" à com...",www.bbc.com,,0.0653,0.5,0.9,0.2792,https://www.bbc.com/afrique/monde-40802096


### Digest: global headlines
*relevance α=0.6, recency β=0.3, credibility γ=0.1*

**1. [Aberdeen Asset Management reports new business slowdown](https://www.bbc.com/news/uk-scotland-scotland-business-18955127)**  
*www.bbc.com*
- The investment firm said net inflows fell year-on-year by £400m to £300m.
- However, assets under management remained stable at £182.7bn.

**2. [Un accord pétrolier remis en cause au Nigeria](https://www.bbc.com/afrique/region-46341856)**  
*www.bbc.com*
- Les géants italiens et anglo-néerlandais de l'énergie nient tout acte répréhensible.
- Le tribunal de Milan est en train d'évaluer les responsabilités de l'ancien ministre nigérian du pétrole, Dan Etete, dans l'attribution du bloc OPL 245 à Malabu, une société qu'il contrôlait.

**3. [Environnement: "l'humanité vit à crédit" à compter du 2 août](https://www.bbc.com/afrique/monde-40802096)**  
*www.bbc.com*
- Selon l'ONG, ce moment survient de plus en plus tôt chaque année.
- Selon les deux organisations, "des signes encourageants" indiquent cependant "qu'il est possible d'inverser la tendance".

**4. [Trump tariffs will hurt global growth, IMF warns](https://www.bbc.com/news/business-44851548)**  
*www.bbc.com*
- While the IMF has maintained its global growth forecast of 3.9% for this year and next, it has cut the outlook for the UK and the eurozone among others.
- Emerging economies The IMF said it continues to expect emerging markets and developing economies to grow by 4.9% this year.

**5. [UN launches push for net-zero emissions by 2050](https://www.bbc.com/news/uk-scotland-52939325)**  
*www.bbc.com*
- "If Covid-19 has taught us anything it is that society can, where necessary, pull together to address a global challenge." In a virtual launch that was troubled with technical problems, a speech by COP26 president Alok Sharma - who is also the UK business secretary - was cut short.
- "I do believe that COP26 can be that moment when the world unites behind a fairer recovery from the effects of Covid-19, a recovery that ultimately delivers for both our people and of course our planet." One of the world's biggest manufacturers of plane engines, Rolls-Royce, has committed to the campaign.

**6. [N-Dubz rapper took mephedrone](https://www.bbc.com/news/newsbeat-10088828)**  
*www.bbc.com*
- I've learnt my mistake.
- I'll never do such things again." Pictures from CCTV footage filmed in an Essex nightclub last month published in The Sun show the rapper, real name Dino Contostavlos, inhaling a white substance.


# Evaluation (ROUGE & latency)


## ROUGE-1/2/L sanity (tiny sample; robust fallback)

In [4]:
# PURPOSE: Quick quality sanity with ROUGE on a small random set (uses eval_pairs you saved).

try:
    from rouge_score import rouge_scorer
    scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)

    # small sample to keep it fast
    eval_sample = eval_pairs.sample(n=min(150, len(eval_pairs)), random_state=42)
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for _, row in eval_sample.iterrows():
        pred = summarize_text(row["text"], max_words=50)
        ref  = row["summary_ref"]
        s = scorer.score(ref, pred)
        for k in scores:
            scores[k].append(s[k].fmeasure)

    print("Mean ROUGE (F1):", {k: round(float(pd.Series(v).mean()), 4) for k, v in scores.items()})
except Exception as e:
    print("ROUGE not available; skipping. (Install rouge-score if internet allowed.)")


ROUGE not available; skipping. (Install rouge-score if internet allowed.)


## Latency (retrieval vs. digest)

In [5]:
# PURPOSE: Simple latency table for 3 topics.

import time
topics = ["AI in healthcare", "économie française", "football"]
rows = []
for t in topics:
    t0 = time.time(); _ = tool_focus(t, top_n=3); t1 = time.time()
    _ = tool_digest(t, n_items=6); t2 = time.time()
    rows.append({"topic": t, "focus_ms": int(1000*(t1-t0)), "digest_ms": int(1000*(t2-t1))})
latency_df = pd.DataFrame(rows)
display(latency_df)


,topic,focus_ms,digest_ms
0,AI in healthcare,15,171
1,économie française,14,176
2,football,13,634
